In [7]:
import pandas as pd

# Se carga el archivo para poder leer "manipular"
data_frame = pd.read_csv('server_logs.csv')

In [8]:
data_frame

,timestamp_event,received_at,service_name,severity,message,trace_id,request_id,method,endpoint,status_code,latency_ms,host,env,region,log_type
0,2026-01-10T00:02:39.029160Z,2026-01-10T00:02:39.097160Z,orders-service,INFO,Request completed,bd8e6ebe717b4c43ba5e72e1668641a8,e6b3009adcad,GET,/orders/create,200,99,orders-service-pod-03,prod,sa-east-1,request
1,2026-01-10T00:02:46.081021Z,2026-01-10T00:02:46.196021Z,api-gateway,INFO,Background job completed,40ac5ff97bae43d5b8045484725e0aca,e2aa1cccd1cf,GET,/health,200,122,api-gateway-pod-01,prod,sa-east-1,request
2,2026-01-10T00:04:01.648849Z,2026-01-10T00:04:01.718849Z,notification-service,WARN,Rate limit nearing threshold,c048b378759a4c54a6f7d7251a6acc88,c74106ca8581,POST,/notify/sms,200,646,notification-service-pod-03,prod,sa-east-1,request
3,2026-01-10T00:05:08.148346Z,2026-01-10T00:05:08.236346Z,api-gateway,INFO,Background job completed,072169f708c84227986bfda9f9657bdc,bf10fa3609bd,GET,/checkout,200,127,api-gateway-pod-03,prod,sa-east-1,request
4,2026-01-10T00:05:19.590837Z,2026-01-10T00:05:19.632837Z,inventory-service,INFO,Health check OK,e76a214131d24a4ea1acf389362e34cb,f672d669eda7,GET,/inv/release,200,144,inventory-service-pod-02,prod,sa-east-1,request
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5790,2026-01-12T23:58:01.010270Z,2026-01-12T23:58:01.081270Z,orders-service,INFO,Background job completed,8a946bacaec44f06b6eb32413cca48b7,0c9c614266b0,GET,/orders/status,200,80,orders-service-pod-02,prod,sa-east-1,request
5791,2026-01-12T23:58:11.592205Z,2026-01-12T23:58:11.654205Z,payment-service,INFO,Health check OK,08c6d50740004e7ea14901f808070245,64f505797722,POST,/pay/charge,200,180,payment-service-pod-02,prod,sa-east-1,request
5792,2026-01-12T23:58:43.915940Z,2026-01-12T23:58:43.984940Z,auth-service,INFO,Health check OK,2d5f02b1c9224094b8fee9027d91922e,388cf2826584,POST,/auth/login,200,122,auth-service-pod-02,prod,sa-east-1,request
5793,2026-01-12T23:59:23.187914Z,2026-01-12T23:59:23.227914Z,auth-service,INFO,Health check OK,d8a425cfaa134d77b36dd863bce2b93e,95525a172704,GET,/auth/refresh,200,158,auth-service-pod-01,prod,sa-east-1,request


In [9]:

#print(data_frame.to_string())   ->  Print de verificacion

# Convertir el timestamp_event en objetos tipo datetime
data_frame['timestamp_event'] = pd.to_datetime(data_frame['timestamp_event'])
# print(data_frame['timestamp_event'])  -> Print de verificacion

# Creamos una columna para verificar "Bad Event"
data_frame['malo'] = (data_frame['severity'].isin(['ERROR', 'CRITICAL'])) | (data_frame["status_code"] >= 500)
#print(data_frame['malo'])   #->  Print de verificacion

#Definiciones Operativas - Time Window(bin) - Agrupacion en ventanas de 5 minutos
data_frame_resumen = data_frame.set_index('timestamp_event').resample('5min').agg(
    total_events = ('service_name', 'count'),
    bad_events = ('malo', 'sum'),
    average_latency = ('latency_ms', 'mean')     # Averiguar bien en que etapa y para que se usa
)

# Calculo del Bad Rate
data_frame_resumen['badRate'] = data_frame_resumen['bad_events'] / data_frame_resumen['total_events']
# print(data_frame_resumen['Bad Rate'])   -> Print de verificacion

# Deteccion momento critico, total_events (minimo 20), ordenamo por el peor Bad Rate
peores_ventanas = data_frame_resumen[data_frame_resumen['total_events'] >= 20].sort_values('badRate', ascending=False)
inicio_momento_critico = peores_ventanas.index[0]

# Diagnostico de lo ocurrido en los 5min
# Se filtra el data_frame original, solo para ese momento
data_inicident =  data_frame[(data_frame['timestamp_event'] >= inicio_momento_critico) &
                            (data_frame['timestamp_event'] < (inicio_momento_critico + pd.Timedelta(minutes=5)))]

# Comparacion del incidente vs Baseline(baseline es todo lo que NO es el momento critico, incidente)
data_baseline = data_frame[data_frame['timestamp_event'] != inicio_momento_critico] 


Total de logs

In [10]:

total_logs = data_frame.shape[0]
#print("Cantidad total de logs:", total_logs)     -> print de verificacion

Por severidad

In [11]:
#data_frame['severity'].unique()
#data_frame.groupby('severity').size()

tiposDeSeveridad = data_frame['severity'].value_counts()
severidadMasComun = data_frame['severity'].value_counts().idxmax()
#print(tiposDeSeveridad, severidadMasComun)     -> Print de verificacion


Servicio con mas logs

In [12]:
# Conteo de servicios y cantidad de apariciones
conteoLogs = data_frame['service_name'].value_counts()
servicioMasLogs = data_frame['service_name'].value_counts().idxmax()
#print(conteoLogs, servicioMasLogs)         -> Print de verificacion

Servicio con menos logs

In [13]:
servicioMenosLogs = data_frame['service_name'].value_counts().idxmin()  
#print(servicioMenosLogs)      -> Print de verificacion

Mensaje mas frecuente

In [14]:
mensajeMasRepetido = data_frame['message'].value_counts().idxmax()
print(mensajeMasRepetido)

Health check OK


Mensaje "malo" mas frecuente

In [ ]:
mensaje_malo_mas_repetido = data_frame[data_frame['malo'] == True].value_counts('message').idxmax()
mensaje_malo_mas_repetido


'Order creation failed - inventory lock timeout'